# 5 · Courbes et surfaces — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch05_courbes/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch05_courbes/03_notebook.ipynb`, créée par `python tools/start_chapter.py 5`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 5.1 à 5.5 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 5.11 | Dérivées numériques : première et seconde | 🔨 | ★★ | 25 |
|  | 5.12 | Quel pas h choisir ? Prédire la courbe d'erreur | 🔮 | ★★ | 15 |
|  | 5.13 | Erreur de troncature contre erreur d'arrondi | 🔬 | ★★ | 25 |
|  | 5.14 | Les maxima des cycles solaires | 🔨 | ★★ | 30 |
| B | 5.15 | numerical_gradient sur la vallée de Rosenbrock | 🔨 | ★★ | 25 |
|  | 5.16 | Le gradient qui abîme son entrée | 🐛 | ★★ | 20 |
|  | 5.17 | Lire des lignes de niveau : où pointe le gradient ? | 📈 | ★★ | 20 |
| C | 5.18 | gradient_descent, et sa version qui monte | 🔨 | ★★ | 30 |
|  | 5.19 | Learning rate sur un bol : trop petit, juste, trop grand | 🔬 | ★★ | 25 |
|  | 5.20 | Démarrer pile sur un point selle | 🔮 | ★★ | 15 |
|  | 5.21 | Le même gradient avec torch.autograd | 📦 | ★★ | 20 |
| D | 5.22 | L'eau qui descend la surface (figure 5.18) | 🎨 | ★★ | 30 |
|  | 5.23 | Tests de propriétés paramétrés avec pytest | 🛠️ | ★★ | 20 |
|  | 5.24 | Minimum, maximum, selle ou plat : classify_critical_point | 🔨 | ★★★ | 35 |
|  | 5.25 | Atteindre le fond de la vallée de Rosenbrock | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")  # your own library

## Objectifs et rappel express

- Estimer une dérivée et un gradient par différences finies, et choisir le pas $h$ (troncature contre arrondi).
- Repérer les extrema d'une courbe échantillonnée, et classer les points critiques d'une surface (minimum, maximum, point selle).
- Écrire et régler une descente de gradient : le learning rate, les courbures, les points selles.
- Comparer le gradient numérique et la différentiation automatique (`torch.autograd`).

**Rappel express.** Pente centrée $\frac{f(x + h) - f(x - h)}{2h}$ (erreur en $h^2$), avant $\frac{f(x + h) - f(x)}{h}$ (erreur en $h$) ; dérivée seconde $\frac{f(x + h) - 2f(x) + f(x - h)}{h^2}$ ; gradient $\nabla f = \left(\frac{\partial f}{\partial x_1}, \ldots, \frac{\partial f}{\partial x_n}\right)$, perpendiculaire aux lignes de niveau, vers la plus grande montée ; descente $\mathbf{x} \leftarrow \mathbf{x} - \eta\,\nabla f(\mathbf{x})$. En Python : `np.array(x, dtype=float)` (une copie), `np.linalg.norm`, `np.meshgrid`, `ax.contour`, `ax.quiver`, `torch.tensor(..., requires_grad=True)` et `.backward()`.

## Partie 0 · Vérifier tes exercices papier (✏️ 5.1 à 5.5)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[1, 2, 3]` ou `True`), pas l'expression Python, sinon tu ne vérifies rien. En Python, le séparateur décimal est un **point** (`0.125`) ; `0,125` sans guillemets serait un couple de deux nombres. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul ; quand l'énoncé dit « valeurs exactes », donne-les sans arrondir. Les réponses pas encore remplies affichent ⏳. Les exercices ∂ 5.6 et 5.7 se corrigent avec `05_solutions.md`.

**Ex 5.1 — La sécante qui se resserre sur la tangente**

In [ ]:
# Ex 5.1: the values you found on paper
answer_5_1a = ...  # a) slope of the symmetric secant for h = 1, 3 decimals
answer_5_1b = ...  # b) slope for h = 0.5, 4 decimals
answer_5_1c = ...  # c) slope for h = 0.1, 5 decimals
answer_5_1d = ...  # d) f'(2), the limit when h tends to 0
answer_5_1e = ...  # e) slope of the forward secant between 2 and 2.1, 4 decimals
answer_5_1f = ...  # f) error of the forward slope / error of the symmetric slope, 1 decimal

for letter, answer in zip("abcdef", [answer_5_1a, answer_5_1b, answer_5_1c, answer_5_1d, answer_5_1e, answer_5_1f]):
    wb.check(f"5.1{letter}", answer)

**Ex 5.2 — Gradient à la main et direction de plus grande pente**

In [ ]:
# Ex 5.2: the values you found on paper
answer_5_2a = ...  # a) f(P)
answer_5_2b = ...  # b) the gradient [df/dx, df/dy] at P, a list
answer_5_2c = ...  # c) its norm, 3 decimals
answer_5_2d = ...  # d) the unit vector of steepest DESCENT, a list (3 decimals)
answer_5_2e = ...  # e) the slope of f at P in the direction u = (0.6, 0.8), 1 decimal
answer_5_2f = ...  # f) a unit vector with zero slope, first component positive, a list (3 decimals)

for letter, answer in zip("abcdef", [answer_5_2a, answer_5_2b, answer_5_2c, answer_5_2d, answer_5_2e, answer_5_2f]):
    wb.check(f"5.2{letter}", answer)

**Ex 5.3 — Trois pas de descente de gradient à la main**

In [ ]:
# Ex 5.3: the values you found on paper
answer_5_3a = ...  # a) f'(5)
answer_5_3b = ...  # b) [x1, x2, x3]
answer_5_3c = ...  # c) [f(x0), f(x1), f(x2), f(x3)]
answer_5_3d = ...  # d) ascent on g(x) = 4x - x², [x1, x2, x3]
answer_5_3e = ...  # e) with eta = 0.5: [x1, x2]
answer_5_3f = ...  # f) the upper bound of the learning rates that converge
answer_5_3g = ...  # g) the learning rate that reaches the minimum in one step

for letter, answer in zip("abcdefg", [answer_5_3a, answer_5_3b, answer_5_3c, answer_5_3d, answer_5_3e, answer_5_3f, answer_5_3g]):
    wb.check(f"5.3{letter}", answer)

**Ex 5.4 — Tableau de variations : extrema locaux et globaux de x³ − 3x**

In [ ]:
# Ex 5.4: the values you found on paper
answer_5_4a = ...  # a) the points where f'(x) = 0, in increasing order (a list)
answer_5_4b = ...  # b) the values of f at the points of a), same order (a list)
answer_5_4c = ...  # c) f'' at the smaller point of a)
answer_5_4d = ...  # d) the global maximum on [-2.5, 2.5], 3 decimals
answer_5_4e = ...  # e) where it is reached (x)
answer_5_4f = ...  # f) on [-2, 2], the number of points where f reaches its global maximum
answer_5_4g = ...  # g) where f'' = 0 (x)

for letter, answer in zip("abcdefg", [answer_5_4a, answer_5_4b, answer_5_4c, answer_5_4d, answer_5_4e, answer_5_4f, answer_5_4g]):
    wb.check(f"5.4{letter}", answer)

**Ex 5.5 — Point selle : x² − y² vu dans deux directions**

In [ ]:
# Ex 5.5: the values you found on paper
answer_5_5a = ...  # a) the gradient at (0, 0), a list
answer_5_5b = ...  # b) [second derivative along x, along y]
answer_5_5c = ...  # c) second derivative along theta = 30 degrees
answer_5_5d = ...  # d) the angle in [0, 90] degrees where it is zero
answer_5_5e = ...  # e) the point after two steps from (0.5, 0.1) with eta = 0.25, a list
answer_5_5f = ...  # f) f at that point, 3 decimals
answer_5_5g = ...  # g) the point after three steps from (0.5, 0), a list

for letter, answer in zip("abcdefg", [answer_5_5a, answer_5_5b, answer_5_5c, answer_5_5d, answer_5_5e, answer_5_5f, answer_5_5g]):
    wb.check(f"5.5{letter}", answer)

## Partie A · Dériver sur ordinateur : le pas h et les cycles solaires

Fiche §5.2 et §5.3, puis « au-delà du livre » (1) et (2). Tu écris tes premières fonctions de `mylearn.calculus` (les dérivées numériques), tu prévois puis mesures l'effet du pas $h$, et tu repères les cycles solaires avec les extrema locaux. La cellule ci-dessous charge les outils de tout le notebook : exécute-la d'abord.

In [ ]:
# Tools for the notebook exercises (parts A to D)
import matplotlib.pyplot as plt
import numpy as np
import pytest
import torch


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def run_calculus_tests(keyword, impl="learner"):
    """Run the tests of mylearn.calculus selected by `keyword` (on YOUR code by default)."""
    command = [sys.executable, "-m", "pytest", "tests/test_ch05_calculus.py", "-k", keyword, "-q",
               "-p", "no:cacheprovider", "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix("FAILED tests/test_ch05_calculus.py::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


def error_name(func, *args, **kwargs):
    """Name of the exception raised by func(*args, **kwargs), or "no error"."""
    try:
        func(*args, **kwargs)
    except NotImplementedError:
        raise
    except Exception as error:  # noqa: BLE001 - we want the name of whatever is raised
        return type(error).__name__
    return "no error"


def rosenbrock(v):
    """Rosenbrock function (a = 1, b = 100) at the point v = [x, y]: a float, minimum 0 at (1, 1)."""
    return float(wb.synth.rosenbrock(v[0], v[1]))


def rosenbrock_gradient(v):
    """Its exact gradient at v = [x, y], from the formula (∂ 5.7): the array [df/dx, df/dy]."""
    return np.array(wb.synth.rosenbrock_grad(v[0], v[1]))


print("tools ready: rosenbrock(v) =", rosenbrock([-1.5, 2.0]), "at (-1.5, 2)")

### Ex 5.11 — Dérivées numériques : première et seconde 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire la pente par différences finies (avant, arrière, centrée) et la dérivée seconde, avec leurs contrôles.  
**Prérequis :** Ex 5.1 · fiche §5.3, au-delà du livre (1) · **Fil rouge :** synthétique · **mylearn :** `calculus.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/calculus.py` (créé par `python tools/start_chapter.py 5`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy est permis (`np.asarray`, `np.ndim`, `np.array`, `np.zeros_like`, `np.eye`, `np.linalg.norm`…), SciPy et PyTorch non : `scipy.signal.argrelextrema`, `torch.autograd` et `torch.optim.SGD` sont les **oracles** des tests. Écris une petite fonction d'aide, par exemple `_check_step(h)`, qui lève une `ValueError` si `h` n'est pas strictement positif : les trois fonctions qui prennent un pas `h` l'appelleront. La cellule de vérification recharge ta librairie, vérifie quelques valeurs, puis lance les tests de tes fonctions ; `python -m pytest tests/test_ch05_calculus.py -q` les lance tous dans un terminal.

Écris `numerical_derivative(f, x, h=1e-5, method="central")` et `second_derivative(f, x, h=1e-4)` (lis leurs docstrings) :
- lève une `ValueError` si `h` n'est pas strictement positif, ou si `method` n'est pas `"central"`, `"forward"` ou `"backward"` ;
- applique la formule demandée : elle marche telle quelle sur un tableau de points, pourvu que `f` accepte les tableaux (comme `np.sin`) ;
- pour un `x` scalaire (`np.ndim(x) == 0`), renvoie un `float` Python ; sinon (un tableau ou une liste), convertis d'abord `x` avec `np.asarray(x, dtype=float)`, et renvoie un tableau de flottants de la même forme.

Vérifications sur $f(x) = x\,e^{-x}$ (`f_11`), au point $x = 0{,}2$. La cellule de vérification appelle tes fonctions :
a) la pente **centrée**, avec $h = 0{,}01$ ;
b) la pente **avant**, avec le même pas $h = 0{,}01$ ;
c) la dérivée seconde, avec le pas par défaut ;
puis deux contrôles d'erreurs et les tests des deux fonctions.

Dans tes notes : la dérivée exacte est $f'(x) = (1 - x)\,e^{-x}$. Compare-lui a) et b) : de combien chacune s'en écarte-t-elle, avec le même pas ? Même question pour c), avec $f''(x) = (x - 2)\,e^{-x}$.

In [ ]:
def f_11(x):
    """x e^(-x): works on numbers and on NumPy arrays."""
    return x * np.exp(-x)


X_11 = 0.2

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.11"):
    wb.check("5.11a", mylearn.calculus.numerical_derivative(f_11, X_11, h=0.01), computed=True)
    wb.check("5.11b", mylearn.calculus.numerical_derivative(f_11, X_11, h=0.01, method="forward"), computed=True)
    wb.check("5.11c", mylearn.calculus.second_derivative(f_11, X_11), computed=True)
    verdict("5.11", error_name(mylearn.calculus.numerical_derivative, f_11, X_11, h=0.0) == "ValueError",
            "un pas nul est refusé (ValueError).",
            "numerical_derivative(f_11, 0.2, h=0.0) doit lever une ValueError : le pas doit être strictement positif.")
    verdict("5.11", error_name(mylearn.calculus.numerical_derivative, f_11, X_11, method="middle") == "ValueError",
            "une méthode inconnue est refusée (ValueError).",
            "numerical_derivative(f_11, 0.2, method=\"middle\") doit lever une ValueError : la méthode est inconnue.")
    run_calculus_tests("test_numerical_derivative_ or test_second_derivative_")

### Ex 5.12 — Quel pas h choisir ? Prédire la courbe d'erreur 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir comment l'erreur d'une différence finie varie avec le pas $h$, de $10^{-1}$ à $10^{-15}$.  
**Prérequis :** Ex 5.11 · fiche, au-delà du livre (1) · fais ta prédiction **avant** de lire « Au-delà du livre (2) », la question 7 de ∂ 5.6 et son corrigé · **Fil rouge :** synthétique · **Parcours :** M, C

On estime la pente de $f(x) = x^3 - 2x$ (`f_12`) en $x = 2{,}2$, où elle vaut exactement $f'(2{,}2) = 12{,}52$, avec les pas $h = 10^{-k}$ pour $k = 1, 2, \ldots, 15$, par la différence centrée et par la différence avant. Pour chaque pas, on mesure l'erreur $|\text{estimation} - 12{,}52|$.

**Sans rien exécuter**, prévois :
a) `prediction_5_12a` : le $k$ du pas qui donne la plus petite erreur pour la différence **centrée** (un entier de 1 à 15) ;
b) `prediction_5_12b` : le même, pour la différence **avant** ;
c) `prediction_5_12c` : pour la différence centrée, l'erreur avec $h = 10^{-15}$ est-elle plus grande qu'avec $h = 10^{-1}$ ? (`True` ou `False`) ;
d) `prediction_5_12d` : chacune à son meilleur pas, la différence centrée est-elle plus précise que la différence avant ? (`True` ou `False`).

Écris ton hypothèse (cellule 📝), puis tes quatre prédictions. Ensuite seulement, exécute l'**Expérience**, qui trace les deux courbes d'erreur. Rien n'est vérifié automatiquement ici : compare toi-même, puis note ce que tu retiens ; l'exercice suivant mesure la même courbe avec ta fonction.

In [ ]:
def f_12(x):
    """x³ - 2x, written with products only: the same bits on every computer."""
    return x * x * x - 2 * x


def df_12(x):
    """Its exact derivative, 3x² - 2."""
    return 3 * x * x - 2


X_12 = 2.2                     # f'(2.2) = 12.52
KS_12 = np.arange(1, 16)       # the steps h = 10**-k, for k = 1, 2, ..., 15

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_5_12a = ...   # an integer from 1 to 15
prediction_5_12b = ...   # an integer from 1 to 15
prediction_5_12c = ...   # True or False
prediction_5_12d = ...   # True or False

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_5_12a, prediction_5_12b, prediction_5_12c, prediction_5_12d]):
    print("⏳ Ex 5.12 : écris d'abord tes quatre prédictions, puis relance cette cellule.")
else:
    steps_12 = 10.0 ** -KS_12
    central_12 = [abs((f_12(X_12 + h) - f_12(X_12 - h)) / (2 * h) - df_12(X_12)) for h in steps_12]
    forward_12 = [abs((f_12(X_12 + h) - f_12(X_12)) / h - df_12(X_12)) for h in steps_12]
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.loglog(steps_12, central_12, "o-", label="central difference")
    ax.loglog(steps_12, forward_12, "s-", label="forward difference")
    ax.invert_xaxis()                                       # h decreases from left to right
    ax.set(xlabel="step h (log scale, decreasing)", ylabel="error |estimate - f'(2.2)| (log scale)")
    ax.legend()
    plt.show()
    print(f"your predictions: a) k = {prediction_5_12a}   b) k = {prediction_5_12b}   "
          f"c) {prediction_5_12c}   d) {prediction_5_12d}")

📝 **Ce que je retiens** (comparaison avec mes prédictions ; pourquoi la courbe remonte-t-elle à droite ? pourquoi pas au même endroit pour les deux différences ?) : …

### Ex 5.13 — Erreur de troncature contre erreur d'arrondi 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer les deux erreurs d'une différence finie, et retrouver le meilleur pas en minimisant un modèle de l'erreur.  
**Prérequis :** Ex 5.12 · ∂ 5.6 · fiche, au-delà du livre (2) · **Fil rouge :** synthétique · **Parcours :** M, C

Mesure maintenant la courbe de 5.12 avec **ta** fonction `numerical_derivative`. Écris `errors_13(f, df, x, ks, method)`, qui renvoie le tableau des erreurs $|\text{pente estimée} - f'(x)|$ pour les pas $h = 10^{-k}$, $k$ parcourant `ks` (`df` est la dérivée exacte, une fonction ; attention, `10 ** -k` refuse un exposant entier négatif de NumPy : écris `10.0 ** -k`). La cellule de vérification l'appelle sur `f_12` en $x = 2{,}2$, pour $k = 1, \ldots, 15$, et en tire :
a) le $k$ du meilleur pas, pour la différence centrée ;
b) le même, pour la différence avant ;
c) la **pente** de la courbe d'erreur centrée entre $h = 10^{-1}$ et $h = 10^{-3}$, en coordonnées logarithmiques : $\frac{\log_{10} e(10^{-1}) - \log_{10} e(10^{-3})}{\log_{10} 10^{-1} - \log_{10} 10^{-3}}$ ;
d) la même pente, pour la différence avant ;
puis elle trace tes deux courbes.

Ensuite, un modèle simple de l'erreur totale. Avec un pas $h$, l'erreur de troncature de la différence centrée se comporte comme $h^2$, et son erreur d'arrondi comme $\frac{\varepsilon}{h}$, où $\varepsilon \approx 2{,}2 \times 10^{-16}$ (`np.finfo(float).eps`) ; pour la différence avant, comme $h$ et $\frac{\varepsilon}{h}$ (∂ 5.6, question 7 ; on oublie les constantes). Le meilleur pas minimise la somme : annule sa dérivée par rapport à $h$ (§5.3).
e) `log_h_central_13` : $\log_{10}$ du pas qui minimise $h^2 + \frac{\varepsilon}{h}$ (2 décimales) ;
f) `log_h_forward_13` : $\log_{10}$ du pas qui minimise $h + \frac{\varepsilon}{h}$ (2 décimales).

Dans tes notes : que disent les pentes c) et d) de la façon dont l'erreur de troncature diminue ? Le modèle de e) et f) retrouve-t-il les meilleurs pas mesurés en a) et b) ? Pourquoi la partie des petits pas est-elle si irrégulière ?

In [ ]:
def errors_13(f, df, x, ks, method):
    """|numerical_derivative(f, x, h=10**-k, method) - df(x)| for every k of ks, with YOUR function (an array)."""
    raise NotImplementedError("errors_13() is not written yet")


log_h_central_13 = ...   # e)
log_h_forward_13 = ...   # f)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.13"):
    central_13 = np.asarray(errors_13(f_12, df_12, X_12, KS_12, "central"), dtype=float)
    forward_13 = np.asarray(errors_13(f_12, df_12, X_12, KS_12, "forward"), dtype=float)
    wb.check("5.13a", int(KS_12[np.argmin(central_13)]), computed=True)
    wb.check("5.13b", int(KS_12[np.argmin(forward_13)]), computed=True)
    wb.check("5.13c", (np.log10(central_13[0]) - np.log10(central_13[2])) / 2, computed=True)
    wb.check("5.13d", (np.log10(forward_13[0]) - np.log10(forward_13[2])) / 2, computed=True)
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    steps_13 = 10.0 ** -KS_12
    eps_13 = np.finfo(float).eps
    ax.loglog(steps_13, central_13, "o-", label="central difference (your function)")
    ax.loglog(steps_13, forward_13, "s-", label="forward difference (your function)")
    ax.loglog(steps_13, steps_13 ** 2 + eps_13 / steps_13, "--", color="C0", alpha=0.6, label="model h² + ε/h")
    ax.loglog(steps_13, steps_13 + eps_13 / steps_13, "--", color="C1", alpha=0.6, label="model h + ε/h")
    ax.invert_xaxis()
    ax.set(xlabel="step h (log scale, decreasing)", ylabel="error (log scale)")
    ax.legend(fontsize=8)
    plt.show()
wb.check("5.13e", log_h_central_13)
wb.check("5.13f", log_h_forward_13)

### Ex 5.14 — Les maxima des cycles solaires 🔨 ★★ ⏱️ 30 min
**Objectif :** repérer les extrema locaux d'une série échantillonnée, et mesurer l'effet du lissage et de la largeur de la zone comparée.  
**Prérequis :** Ex 5.11 · Ex 1.12 (taches solaires) · fiche §5.2, §5.3 · livre §5.3 (figures 5.5 et 5.6) · **Fil rouge :** taches solaires · **mylearn :** `calculus.py` · **Parcours :** C

> **mylearn** : dans `mon_travail/mylearn/calculus.py`, mêmes règles qu'en 5.11 (NumPy permis, SciPy et PyTorch non). Enregistre, puis relance la cellule de vérification.

Au ch. 1 (1.12), un outil fourni repérait les sommets de la série lissée des taches solaires. Écris le tien, `find_local_extrema(y, order=1)` (lis sa docstring) : l'échantillon $i$ est un minimum local s'il est **strictement** plus petit que chacun de ses voisins, jusqu'à `order` échantillons de chaque côté (un maximum : strictement plus grand). Les voisins qui tomberaient hors du tableau sont ignorés, mais le premier et le dernier échantillon ne sont jamais des extrema. Renvoie deux tableaux d'indices entiers, triés : les minima, puis les maxima. Lève une `ValueError` si `y` n'est pas à une dimension, ou si `order` n'est pas un entier au moins égal à 1.

C'est la définition « par voisinage » de la fiche (§5.3), pour une courbe échantillonnée : `order` règle la largeur du voisinage. Avec `order=1`, on retrouve les points où la série change de sens, ceux où s'arrête la marche du livre (figures 5.5 et 5.6), sauf sur un palier de valeurs égales (question e). Les données : `raw_14`, les nombres mensuels de taches depuis 1749, et `smooth_14`, leur moyenne mobile **centrée** sur 13 mois (le mois, les 6 précédents et les 6 suivants), sans ses valeurs manquantes ; `dates_14` donne `[année, mois]` pour chaque valeur de `smooth_14`, et `years_14` l'année décimale. La cellule de vérification appelle ta fonction :
a) le nombre de maxima locaux de la série **brute**, avec `order=1` ;
b) le nombre de minima de la série lissée, avec `order=6` ;
c) avec `order=60` (cinq ans de chaque côté), sur la série lissée : `[année, mois]` du dernier minimum, puis du dernier maximum (un tableau 2 × 2) ;
d) avec `order=60`, la plus longue durée entre deux minima successifs, en années ;
puis les tests de `find_local_extrema`, et une figure.

e) `zero_months_14` : le nombre de mois où la série lissée vaut **exactement** 0.

Dans tes notes : la durée de d) correspond-elle à un vrai cycle ? Que s'est-il passé à cette époque, et pourquoi ta fonction ne voit-elle pas le minimum qui manque ? Pourquoi le dernier maximum de c) est-il encore incertain ?

In [ ]:
sun_14 = wb.datasets.load_sunspots(definitive_only=True)        # monthly sunspot numbers since 1749 (ch. 1)
raw_14 = sun_14["sunspots"].to_numpy()
centred_14 = sun_14["sunspots"].rolling(13, center=True).mean()  # the month, the 6 before and the 6 after
kept_14 = centred_14.notna().to_numpy()                          # the first 6 and the last 6 months have no value
smooth_14 = centred_14.to_numpy()[kept_14]
years_14 = sun_14["decimal_year"].to_numpy()[kept_14]
dates_14 = sun_14[["year", "month"]].to_numpy()[kept_14]         # [year, month] of every value of smooth_14
print(f"{len(raw_14)} months; {len(smooth_14)} smoothed values, from {dates_14[0].tolist()} to {dates_14[-1].tolist()}")

In [ ]:
zero_months_14 = ...   # e)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.14"):
    wb.check("5.14a", len(mylearn.calculus.find_local_extrema(raw_14)[1]), computed=True)
    wb.check("5.14b", len(mylearn.calculus.find_local_extrema(smooth_14, order=6)[0]), computed=True)
    minima_14, maxima_14 = mylearn.calculus.find_local_extrema(smooth_14, order=60)
    wb.check("5.14c", [dates_14[minima_14[-1]], dates_14[maxima_14[-1]]], computed=True)
    wb.check("5.14d", np.max(np.diff(years_14[minima_14])), computed=True)
    run_calculus_tests("test_find_local_extrema_")
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot(years_14, smooth_14, lw=1.2, label="13-month centred mean")
    ax.plot(years_14[minima_14], smooth_14[minima_14], "v", label="minima (order 60)")
    ax.plot(years_14[maxima_14], smooth_14[maxima_14], "^", label="maxima (order 60)")
    ax.set(xlabel="year", ylabel="sunspots")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()
wb.check("5.14e", zero_months_14)

## Partie B · Le gradient : le calculer, le déboguer, le lire sur une carte

Fiche §5.4. Tu écris le gradient numérique et tu l'essaies sur la vallée de Rosenbrock et sur une matrice de poids, tu débogues la version d'un collègue, puis tu lis le gradient sur une carte de lignes de niveau.

### Ex 5.15 — numerical_gradient sur la vallée de Rosenbrock 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire le gradient numérique d'une fonction de plusieurs variables, pour un point de forme quelconque, sans toucher au point reçu.  
**Prérequis :** Ex 5.11 · ✏️ 5.2 · fiche §5.4 (calculer un gradient sur ordinateur) · **Fil rouge :** Rosenbrock · **mylearn :** `calculus.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/calculus.py`, mêmes règles qu'en 5.11 (NumPy permis, SciPy et PyTorch non). Enregistre, puis relance la cellule de vérification.

Écris `numerical_gradient(f, x, h=1e-5)` (lis sa docstring) : pour chaque position $i$ de `x`, la différence centrée $\frac{f(\mathbf{x} + h\,\mathbf{e}_i) - f(\mathbf{x} - h\,\mathbf{e}_i)}{2h}$, où $\mathbf{e}_i$ vaut 1 à la position $i$ et 0 ailleurs.
- Travaille sur une **copie en flottants** : `point = np.array(x, dtype=float)` (contrairement à `np.asarray`, `np.array` copie toujours). Le tableau de l'appelant ne doit jamais changer, pas même pendant les appels à `f`.
- `x` peut avoir n'importe quelle forme : une matrice de poids, par exemple. `point.reshape(-1)` en donne une vue à plat (modifier la vue modifie `point`), et `np.zeros_like(point)` prépare un gradient de la même forme.
- Pour chaque position : garde l'ancienne valeur, pose `ancienne + h`, évalue `f`, pose `ancienne - h`, évalue `f`, puis **remets l'ancienne valeur** telle quelle (ne la recalcule pas : `+ h`, `- 2 * h` puis `+ h` ne redonnent pas toujours exactement le même nombre).

La cellule de vérification appelle ta fonction :
a) le gradient de la fonction de Rosenbrock (`rosenbrock`, $a = 1$, $b = 100$) au point $(-0{,}5 ;\ 0{,}5)$ ; elle le compare au gradient exact, `rosenbrock_gradient` ;
b) le gradient de la loss $L(W) = \sum_i \big((W\mathbf{v})_i - t_i\big)^2$ (`loss_15`) par rapport à la **matrice** $W$, au point `W_15` (une matrice 2 × 3) ;
puis le contrôle que `W_15` n'a pas changé, et les tests de `numerical_gradient`.

Dans tes notes : combien d'appels à `loss_15` a-t-il fallu pour b) ? Vérifie b) à la main avec la formule $\nabla_W L = 2\,(W\mathbf{v} - \mathbf{t})\,\mathbf{v}^\top$ (0B).

In [ ]:
W_15 = np.array([[1.0, 0.0, -1.0],
                 [0.5, 2.0, 0.0]])
V_15 = np.array([1.0, 2.0, 3.0])
T_15 = np.array([0.0, 1.0])


def loss_15(W):
    """Sum of the squared errors of W @ V_15 against the targets T_15: a function of the matrix W."""
    return float(np.sum((W @ V_15 - T_15) ** 2))

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.15"):
    grad_15 = np.asarray(mylearn.calculus.numerical_gradient(rosenbrock, [-0.5, 0.5]), dtype=float)
    wb.check("5.15a", grad_15, computed=True)
    exact_15 = rosenbrock_gradient([-0.5, 0.5])
    verdict("5.15", grad_15.shape == (2,) and np.allclose(grad_15, exact_15, rtol=1e-6),
            f"ton gradient numérique colle au gradient exact {exact_15.tolist()}.",
            f"ton gradient numérique s'écarte du gradient exact {exact_15.tolist()}.")
    saved_15 = W_15.copy()
    wb.check("5.15b", mylearn.calculus.numerical_gradient(loss_15, W_15), computed=True)
    verdict("5.15", np.array_equal(W_15, saved_15), "W_15 n'a pas changé.",
            "ta fonction a modifié W_15 : travaille sur une copie, np.array(x, dtype=float).")
    W_15[...] = saved_15                                      # put W_15 back, in case it was modified
    run_calculus_tests("test_numerical_gradient_")

### Ex 5.16 — Le gradient qui abîme son entrée 🐛 ★★ ⏱️ 20 min
**Objectif :** comprendre pourquoi un gradient numérique doit travailler sur une copie en flottants, et corriger la version d'un collègue.  
**Prérequis :** Ex 5.15 · fiche §5.4 et les pièges classiques · **Fil rouge :** Rosenbrock · **Parcours :** C

Un collègue a écrit son propre gradient numérique, `gradient_colleague` (cellule ci-dessous). Il l'a essayé sur des points à virgule, comme $(0{,}1 ;\ 0{,}2)$ : les valeurs sont justes. Puis il lance une descente de gradient depuis le point $(-1, 1)$, écrit sans virgule, `np.array([-1, 1])`. La cellule affiche ce que renvoie sa fonction dans les deux cas.

a) `exact_16` : le vrai gradient de Rosenbrock en $(-1, 1)$ (une liste ; avec `rosenbrock_gradient`, ou à la main avec ∂ 5.7) ;
b) `after_16` : que contient `start_16` **après** l'appel de la cellule ? (une liste de deux entiers : affiche-le) ;
c) `changed_16` : crée un point à virgule, `point_16 = np.array([0.1, 0.2])`, et appelle **une fois** `gradient_colleague(rosenbrock, point_16)`. Combien de ses deux coordonnées ont changé ? (un entier ; compare-les aux valeurs d'origine avec `==` : `print` arrondit l'affichage) ;
d) corrige : écris `gradient_fixed_16(f, x, h=1e-5)`, la fonction du collègue en changeant le moins de lignes possible. La vérification l'appelle sur le point entier $(-1, 1)$ (le gradient obtenu est vérifié) et sur un point à virgule, et contrôle qu'aucun des deux points n'est modifié.

Dans tes notes : explique les deux défauts de la fonction du collègue, et pourquoi il ne voyait rien sur ses points à virgule. Le défaut de c) est minuscule : pourquoi est-il quand même dangereux ?

In [ ]:
def gradient_colleague(f, x, h=1e-5):
    """The colleague's numerical gradient: central differences, one coordinate after the other."""
    x = np.asarray(x)
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x[i] += h
        f_plus = f(x)
        x[i] -= 2 * h
        f_minus = f(x)
        x[i] += h
        grad[i] = (f_plus - f_minus) / (2 * h)
    return grad


print("gradient at (0.1, 0.2):", gradient_colleague(rosenbrock, np.array([0.1, 0.2])))
start_16 = np.array([-1, 1])                    # the starting point of a descent, written without decimal points
print("gradient at (-1, 1):   ", gradient_colleague(rosenbrock, start_16))

In [ ]:
exact_16 = ...     # a) a list
after_16 = ...     # b) a list of two integers
changed_16 = ...   # c) an integer


def gradient_fixed_16(f, x, h=1e-5):
    """The colleague's function, corrected: the same gradient for [-1, 1] as for [-1.0, 1.0], x never modified."""
    raise NotImplementedError("gradient_fixed_16() is not written yet")

In [ ]:
wb.check("5.16a", exact_16)
wb.check("5.16b", after_16)
wb.check("5.16c", changed_16)
with wb.attempt("5.16"):
    integer_16, decimal_16 = np.array([-1, 1]), np.array([0.1, 0.2])
    wb.check("5.16d", gradient_fixed_16(rosenbrock, integer_16), computed=True)
    gradient_fixed_16(rosenbrock, decimal_16)
    verdict("5.16", np.array_equal(integer_16, [-1, 1]) and np.array_equal(decimal_16, [0.1, 0.2]),
            "les points reçus ne changent plus, pas même d'un arrondi.",
            "ta version modifie encore le point reçu : travaille sur une copie en flottants.")

### Ex 5.17 — Lire des lignes de niveau : où pointe le gradient ? 📈 ★★ ⏱️ 20 min
**Objectif :** lire sur une carte de lignes de niveau les extrema, les points selles, la pente et la direction du gradient.  
**Prérequis :** Ex 5.15 · ✏️ 5.2, 5.5 · fiche §5.4 (pente dans une direction, lignes de niveau, points critiques) · **Fil rouge :** synthétique · **Parcours :** R, M

La carte ci-dessous représente $g(x, y) = x^3 - 3x - y^3 + 3y$ par ses **lignes de niveau** : les couleurs claires sont hautes, les sombres basses (voir la barre de couleurs). Six points sont marqués de A à F. **Sans calculer de dérivée** (sauf en f), lis la carte :
a) `max_17` : la lettre du maximum local ;
b) `min_17` : la lettre du minimum local ;
c) `saddles_17` : les lettres des points selles (un ensemble de chaînes, par exemple `{"X", "Y"}`) ;
d) `steeper_17` : en lequel des deux points D et F la pente est-elle la plus forte ? (`"D"` ou `"F"`) ;
e) `uphill_D_17` : en D, vers où la montée est-elle la plus raide ? Une des huit directions de la boussole, le nord en haut : `"N"`, `"NE"`, `"E"`, `"SE"`, `"S"`, `"SO"`, `"O"` (ouest) ou `"NO"` ;
f) `grad_F_17` : enfin, calcule le gradient de $g$ en F (une liste, 2 décimales), avec les coordonnées de F affichées sous la carte.

Une fois tes six réponses écrites, la dernière cellule superpose à la carte les flèches du gradient : regarde leur longueur, leur sens, et leur angle avec les lignes de niveau.

In [ ]:
def g_17(x, y):
    """x³ - 3x - y³ + 3y (works on grids of points)."""
    return x ** 3 - 3 * x - y ** 3 + 3 * y


POINTS_17 = {"A": (1.0, 1.0), "B": (1.0, -1.0), "C": (-1.0, 1.0), "D": (0.0, 0.0), "E": (-1.0, -1.0), "F": (1.8, -0.2)}
X_17, Y_17 = np.meshgrid(np.linspace(-2.2, 2.2, 201), np.linspace(-2.2, 2.2, 201))


def draw_map_17(ax):
    """Filled level lines of g_17 (light = high), the lines themselves and the six points; returns the colour map."""
    colours = ax.contourf(X_17, Y_17, g_17(X_17, Y_17), levels=24, cmap="viridis")
    ax.contour(X_17, Y_17, g_17(X_17, Y_17), levels=colours.levels[::2], colors="white", linewidths=0.5)
    for letter, (x, y) in POINTS_17.items():
        ax.plot(x, y, "o", color="red")
        ax.annotate(letter, (x, y), xytext=(6, 6), textcoords="offset points", color="red", fontsize=13,
                    weight="bold")
    ax.set_aspect("equal")
    ax.set(xlabel="x", ylabel="y")
    return colours


fig, ax = plt.subplots(figsize=(6.5, 5.5))
fig.colorbar(draw_map_17(ax), ax=ax, label="g(x, y)")
plt.show()
print("F =", POINTS_17["F"])

In [ ]:
max_17 = ...        # a) a letter
min_17 = ...        # b) a letter
saddles_17 = ...    # c) a set of letters
steeper_17 = ...    # d) "D" or "F"
uphill_D_17 = ...   # e) "N", "NE", "E", "SE", "S", "SO", "O" or "NO"
grad_F_17 = ...     # f) a list

In [ ]:
for letter, answer in zip("abcdef", [max_17, min_17, saddles_17, steeper_17, uphill_D_17, grad_F_17]):
    wb.check(f"5.17{letter}", answer)

**La carte et ses gradients** : exécute la cellule une fois tes six réponses écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [max_17, min_17, saddles_17, steeper_17, uphill_D_17, grad_F_17]):
    print("⏳ Ex 5.17 : écris d'abord tes six réponses, puis relance cette cellule.")
else:
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    draw_map_17(ax)
    GX_17, GY_17 = np.meshgrid(np.linspace(-2, 2, 15), np.linspace(-2, 2, 15))
    ax.quiver(GX_17, GY_17, 3 * GX_17 ** 2 - 3, -3 * GY_17 ** 2 + 3, color="white")   # the gradient at each point
    ax.set_title("the gradient of g, drawn on its level lines")
    plt.show()

## Partie C · Descendre : le learning rate, les points selles, autograd

Fiche §5.4 et « au-delà du livre » (3). Tu écris la descente de gradient, tu mesures l'effet du learning rate sur un bol, tu prévois ce que fait une descente partie d'un point selle, puis tu compares ton gradient numérique au gradient de PyTorch.

### Ex 5.18 — gradient_descent, et sa version qui monte 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire la descente (et la montée) de gradient, avec le chemin parcouru et un critère d'arrêt.  
**Prérequis :** Ex 5.15 · ✏️ 5.3 · fiche §5.4 (descendre une surface pas à pas) · **Fil rouge :** synthétique · **mylearn :** `calculus.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/calculus.py`, mêmes règles qu'en 5.11 (NumPy permis, SciPy et PyTorch non). Enregistre, puis relance la cellule de vérification.

Écris `gradient_descent(grad, x0, lr=0.01, n_steps=100, tol=None, maximize=False)` (lis sa docstring) :
- lève une `ValueError` si `lr <= 0`, si `n_steps < 0`, ou si `tol` est donné et négatif ;
- pars d'une copie en flottants de `x0`, et range chaque point visité dans une liste (des **copies** : `x.copy()`) ;
- à chaque pas : évalue le gradient au point courant ; si `tol` est donné et que la norme euclidienne de ce gradient (`np.linalg.norm`, sur toutes ses composantes) est **strictement** inférieure à `tol`, arrête-toi sans bouger ; sinon, fais le pas $\mathbf{x} \leftarrow \mathbf{x} - \eta\,\nabla f(\mathbf{x})$, ou $\mathbf{x} + \eta\,\nabla f(\mathbf{x})$ si `maximize` ;
- renvoie le dernier point et le tableau du chemin (`np.array` de la liste), de forme `(nombre de pas faits + 1, *x0.shape)`.

Vérifications sur deux fonctions dont les gradients sont fournis : $q(\mathbf{v}) = (v_0 + 1)^2 + 4\,(v_1 - 0{,}5)^2$, de minimum $(-1 ;\ 0{,}5)$, et $m(\mathbf{v}) = -(v_0 - 1)^2 - 2\,(v_1 + 2)^2$, de maximum $(1 ;\ -2)$. La cellule de vérification appelle ta fonction :
a) le point atteint sur $q$ depuis $(1, 1)$, avec `lr=0.1` et 15 pas ;
b) le nombre de pas faits sur $q$ depuis $(1, 1)$, avec `lr=0.1`, `n_steps=1000` et `tol=1e-3` ;
c) le point atteint en **montant** $m$ depuis $(0, 0)$, avec `lr=0.2` et 3 pas ;
puis deux contrôles, les tests de `gradient_descent`, et la trajectoire de a) sur la carte de $q$.

Dans tes notes : en a), pourquoi $v_1$ est-il déjà au minimum alors que $v_0$ en est encore loin ? Regarde le facteur qui multiplie, à chaque pas, l'écart de chaque coordonnée au minimum (✏️ 5.3).

In [ ]:
def grad_q_18(v):
    """Gradient of q(v) = (v0 + 1)² + 4 (v1 - 0.5)²."""
    return np.array([2 * (v[0] + 1), 8 * (v[1] - 0.5)])


def grad_m_18(v):
    """Gradient of m(v) = -(v0 - 1)² - 2 (v1 + 2)²."""
    return np.array([-2 * (v[0] - 1), -4 * (v[1] + 2)])

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.18"):
    end_18, path_18 = mylearn.calculus.gradient_descent(grad_q_18, [1.0, 1.0], lr=0.1, n_steps=15)
    wb.check("5.18a", end_18, computed=True)
    _, stopped_18 = mylearn.calculus.gradient_descent(grad_q_18, [1.0, 1.0], lr=0.1, n_steps=1000, tol=1e-3)
    wb.check("5.18b", len(stopped_18) - 1, computed=True)
    wb.check("5.18c", mylearn.calculus.gradient_descent(grad_m_18, [0.0, 0.0], lr=0.2, n_steps=3, maximize=True)[0],
             computed=True)
    start_18 = np.array([1.0, 1.0])
    mylearn.calculus.gradient_descent(grad_q_18, start_18, lr=0.1, n_steps=5)
    verdict("5.18", np.array_equal(start_18, [1.0, 1.0]), "le point de départ de l'appelant n'a pas bougé.",
            "ta fonction a modifié x0 : pars d'une copie, np.array(x0, dtype=float).")
    verdict("5.18", error_name(mylearn.calculus.gradient_descent, grad_q_18, [1.0, 1.0], lr=0.0) == "ValueError",
            "un learning rate nul est refusé (ValueError).",
            "gradient_descent(grad_q_18, [1.0, 1.0], lr=0.0) doit lever une ValueError.")
    run_calculus_tests("test_gradient_descent_")
    wb.plot.plot_contour(lambda x, y: (x + 1) ** 2 + 4 * (y - 0.5) ** 2, xlim=(-2, 1.5), ylim=(-0.5, 1.5),
                         path=path_18, log=False, minimum=(-1, 0.5), levels=15, title="15 steps of gradient descent on q")
    plt.show()

### Ex 5.19 — Learning rate sur un bol : trop petit, juste, trop grand 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer l'effet du learning rate sur une surface dont les deux directions n'ont pas la même courbure.  
**Prérequis :** Ex 5.18 · Ex 1.17 · ✏️ 5.3 · fiche §5.4 (un learning rate pour plusieurs courbures) · **Fil rouge :** synthétique · **Parcours :** C

Le bol $b(\mathbf{v}) = v_0^2 + 10\,v_1^2$ est dix fois plus courbé selon $v_1$ que selon $v_0$. Avec ta fonction `gradient_descent`, descends-le depuis $(2, 1)$, 50 pas, avec chacun des learning rates de `LRS_19` : 0,01 ; 0,05 ; 0,09 ; 0,1 et 0,11.

Écris `distances_19(lr)` : la distance au minimum $(0, 0)$ de chacun des 51 points du chemin, départ compris (un tableau de 51 nombres ; `np.linalg.norm(..., axis=1)`). La vérification contrôle tes distances avec la formule exacte (à chaque pas, $v_0$ est multiplié par $1 - 2\eta$ et $v_1$ par $1 - 20\eta$, ✏️ 5.3), puis trace les distances pas à pas, en échelle logarithmique, et les cinq trajectoires sur la carte du bol.

Réponds dans la cellule 📝 :
- quels learning rates convergent ? Lequel est le plus rapide, et pourquoi n'est-ce pas le plus grand ?
- que fait la descente avec 0,1 ? avec 0,11 ? Quelle direction fixe le plus grand learning rate possible, et laquelle fixe la vitesse ?
- avec 0,01, combien de pas faudrait-il, à peu près, pour arriver à moins de 0,01 du minimum ?

In [ ]:
def grad_bowl_19(v):
    """Gradient of b(v) = v0² + 10 v1²."""
    return np.array([2 * v[0], 20 * v[1]])


START_19 = [2.0, 1.0]
LRS_19 = [0.01, 0.05, 0.09, 0.1, 0.11]

In [ ]:
def distances_19(lr):
    """Distances to (0, 0) of the 51 points of 50 steps of YOUR gradient_descent on the bowl, from START_19."""
    raise NotImplementedError("distances_19() is not written yet")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.19"):
    curves_19 = {lr: np.asarray(distances_19(lr), dtype=float) for lr in LRS_19}
    exact_19 = {lr: np.hypot(2 * (1 - 2 * lr) ** np.arange(51), (1 - 20 * lr) ** np.arange(51)) for lr in LRS_19}
    right_19 = all(curve.shape == (51,) and np.allclose(curve, exact_19[lr], rtol=1e-9)
                   for lr, curve in curves_19.items())
    verdict("5.19", right_19,
            "tes distances suivent la formule exacte : lis les graphiques, puis réponds dans la cellule 📝.",
            "distances_19(lr) doit renvoyer les 51 distances au minimum (départ compris) de 50 pas de gradient_descent "
            "depuis START_19 ; les graphiques s'afficheront ensuite.")
    if right_19:
        steps_19 = np.arange(51)
        colours_19 = dict(zip(LRS_19, ["#f58518", "#e45756", "#54a24b", "#000000", "#b279a2"]))   # one colour per lr
        fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12.5, 4.3))
        for lr, curve in curves_19.items():
            ax_left.semilogy(steps_19, curve, color=colours_19[lr], label=f"lr = {lr}")
        ax_left.set(xlabel="step", ylabel="distance to the minimum (log scale)")
        ax_left.legend(fontsize=8)
        wb.plot.plot_contour(lambda x, y: x ** 2 + 10 * y ** 2, xlim=(-2.5, 2.5), ylim=(-1.6, 1.6), ax=ax_right, log=False,
                             minimum=(0, 0), resolution=150, levels=15)
        for lr in LRS_19[:4]:                                     # lr = 0.11 leaves the frame: see the left panel
            path = mylearn.calculus.gradient_descent(grad_bowl_19, START_19, lr=lr, n_steps=50)[1]
            ax_right.plot(path[:, 0], path[:, 1], ".-", color=colours_19[lr], lw=1, ms=4, label=f"lr = {lr}")
        ax_right.set(xlim=(-2.5, 2.5), ylim=(-1.6, 1.6), title="the first four descents")
        ax_right.set_aspect("equal")
        ax_right.legend(fontsize=7, loc="lower left")
        plt.show()

📝 **Mes réponses** (learning rates qui convergent, le plus rapide ; 0,1 et 0,11 ; la direction qui fixe le learning rate maximal, celle qui fixe la vitesse ; le nombre de pas avec 0,01) : …

### Ex 5.20 — Démarrer pile sur un point selle 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir ce que fait une descente de gradient partie d'un point selle, ou d'un point très proche.  
**Prérequis :** Ex 5.18 · ✏️ 5.5 · fiche §5.4 (les points où le gradient s'annule, 🕰️ points selles) · **Fil rouge :** synthétique · **Parcours :** C

La surface $s(x, y) = x^2 + \frac{y^4}{4} - \frac{y^2}{2}$ a un point selle en $(0, 0)$ et deux minima, en $(0, 1)$ et $(0, -1)$ ; son gradient vaut $(2x,\ y^3 - y)$. On la descend avec `lr=0.1`, pendant 2 000 pas. **Sans rien exécuter**, prévois :
a) `prediction_5_20a` : partie **exactement** de $(0, 0)$, la descente y reste-t-elle ? (`True` ou `False`) ;
b) `prediction_5_20b` : partie de $(0{,}5 ;\ 10^{-6})$, un millionième au-dessus de l'axe des $x$, où finit-elle ? `1` pour $(0, 1)$, `-1` pour $(0, -1)$, `0` si elle reste près de $(0, 0)$ ;
c) `prediction_5_20c` : depuis ce même départ, combien de pas faut-il pour que $|y|$ dépasse 0,5 ? Choisis l'ordre de grandeur : `10`, `100`, `1000` ou `10000`.

Écris ton hypothèse (cellule 📝), puis tes trois prédictions ; la vérification ne regarde que tes prédictions. Ensuite seulement, exécute l'**Expérience** : quatre départs, dont $(0{,}5 ;\ -10^{-6})$ et $(0{,}5 ;\ 10^{-12})$.

In [ ]:
def grad_s_20(v):
    """Gradient of s(x, y) = x² + y⁴/4 - y²/2: a saddle at (0, 0), minima at (0, 1) and (0, -1)."""
    return np.array([2 * v[0], v[1] ** 3 - v[1]])


def descend_20(start, lr=0.1, n_steps=2000):
    """Plain gradient descent on s (no mylearn here): the array of the n_steps + 1 points visited."""
    points = [np.array(start, dtype=float)]
    for _ in range(n_steps):
        points.append(points[-1] - lr * grad_s_20(points[-1]))
    return np.array(points)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_5_20a = ...   # True or False
prediction_5_20b = ...   # 1, -1 or 0
prediction_5_20c = ...   # 10, 100, 1000 or 10000

In [ ]:
wb.check("5.20a", prediction_5_20a)
wb.check("5.20b", prediction_5_20b)
try:                                                        # c) is one of four proposed orders of magnitude
    off_scale_20 = filled(prediction_5_20c) and float(str(prediction_5_20c).replace(",", ".")) not in (10, 100, 1000, 10000)
except ValueError:
    off_scale_20 = False
if off_scale_20:
    print("❌ Ex 5.20c : réponds par l'un des quatre ordres de grandeur proposés : 10, 100, 1000 ou 10000.")
else:
    wb.check("5.20c", prediction_5_20c)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_5_20a, prediction_5_20b, prediction_5_20c]):
    print("⏳ Ex 5.20 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    fig, (ax_y, ax_s) = plt.subplots(1, 2, figsize=(12.5, 3.8))
    for colour, start in enumerate([(0.0, 0.0), (0.5, 1e-6), (0.5, -1e-6), (0.5, 1e-12)]):
        path = descend_20(start)
        beyond = np.flatnonzero(np.abs(path[:, 1]) > 0.5)
        first = int(beyond[0]) if beyond.size else None
        print(f"start {start}: after 2,000 steps ({path[-1, 0]:.3f}, {path[-1, 1]:.3f}); first step with |y| > 0.5: {first}")
        height = path[:, 0] ** 2 + path[:, 1] ** 4 / 4 - path[:, 1] ** 2 / 2
        ax_s.plot(height[:400], color=f"C{colour}", ls="--" if start[1] < 0 else "-", label=f"start {start}")
        if start[1] != 0:                                    # |y| = 0 has no logarithm
            ax_y.semilogy(np.abs(path[:400, 1]), color=f"C{colour}", ls="--" if start[1] < 0 else "-",
                          label=f"start {start}")
    ax_y.set(xlabel="step", ylabel="|y| (log scale)")
    ax_s.set(xlabel="step", ylabel="height s(x, y)")
    ax_y.legend(fontsize=8)
    ax_s.legend(fontsize=8)
    plt.show()
    print(f"your predictions: a) {prediction_5_20a}   b) {prediction_5_20b}   c) {prediction_5_20c}")

Près du point selle, $|y|$ est petit et $y^3$ est négligeable devant $y$ : un pas fait alors $y \leftarrow y - \eta\,(y^3 - y) \approx (1 + \eta)\,y = 1{,}1\,y$.

d) `extra_steps_5_20d` : avec ce facteur, combien de pas de plus faut-il pour que $|y|$ dépasse 0,5 quand le départ est $10^{-12}$ au lieu de $10^{-6}$, un million de fois plus près de l'axe ? (un entier, arrondi au plus proche ; compare-le ensuite à l'expérience)

In [ ]:
extra_steps_5_20d = ...   # d) an integer

In [ ]:
wb.check("5.20d", extra_steps_5_20d)

### Ex 5.21 — Le même gradient avec torch.autograd 📦 ★★ ⏱️ 20 min
**Objectif :** calculer un gradient par différentiation automatique avec PyTorch, le comparer au gradient numérique, et voir ce qu'il donne en un coin.  
**Prérequis :** Ex 5.15 · fiche, au-delà du livre (3) (🧮 PyTorch en cinq lignes) · quiz Q9 · **Fil rouge :** Rosenbrock · **Parcours :** R, C

La fonction `torch_gradient` ci-dessous calcule un gradient par **différentiation automatique** : PyTorch enregistre les opérations faites avec le tenseur `point`, puis `backward()` applique la règle de la chaîne en remontant ces opérations (fiche, au-delà du livre (3)). Il faut lui donner une fonction écrite avec des opérations que PyTorch sait dériver : `+`, `-`, `*`, `/`, `**`, `torch.exp`… mais pas les fonctions de NumPy.

Écris `rosenbrock_torch(p)` : la fonction de Rosenbrock ($a = 1$, $b = 100$) d'un tenseur `p` de deux composantes, `p[0]` et `p[1]`, avec ces opérations. La cellule de vérification l'appelle :
a) le gradient par autograd au point $(-0{,}8 ;\ 0{,}9)$ ; elle le compare ensuite à ton `numerical_gradient`.

La fonction $\max(0, x)$ (la ReLU des réseaux de neurones, ch. 17) a un point anguleux en 0 (quiz Q9) : sa pente vaut 0 à gauche et 1 à droite. PyTorch permet de l'écrire de plusieurs façons. Avec `torch_gradient` et ta fonction `numerical_derivative`, calcule quatre « pentes en 0 » :
b) `corner_21` : la liste [pente de `torch.relu(x)` en 0, pente de `torch.clamp(x, min=0)` en 0, pente de `torch.maximum(x, torch.zeros_like(x))` en 0, différence centrée de `max(0, x)` en 0].

La dernière cellule compare enfin les durées des deux méthodes sur une fonction de 2 000 variables. Dans tes notes : combien d'évaluations de la fonction coûte un gradient numérique pour $n$ variables ? Que penser de la « pente en 0 » de la ReLU ?

In [ ]:
def torch_gradient(f, x):
    """Gradient of f at x by automatic differentiation, in float64 (to compare with a numerical gradient)."""
    point = torch.tensor(x, dtype=torch.float64, requires_grad=True)   # PyTorch records what is done with point
    f(point).backward()                                                # chain rule, from f(point) back to point
    return point.grad.numpy()                                          # df/dpoint, as a NumPy array


POINT_21 = [-0.8, 0.9]

In [ ]:
def rosenbrock_torch(p):
    """Rosenbrock function (a = 1, b = 100) of a tensor p = [x, y], with operations that PyTorch can differentiate."""
    raise NotImplementedError("rosenbrock_torch() is not written yet")


corner_21 = ...   # b) [torch.relu, torch.clamp(x, min=0), torch.maximum(x, torch.zeros_like(x)), central difference of max(0, x)]

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.21"):
    auto_21 = torch_gradient(rosenbrock_torch, POINT_21)
    wb.check("5.21a", auto_21, computed=True)
    numeric_21 = np.asarray(mylearn.calculus.numerical_gradient(rosenbrock, POINT_21), dtype=float)
    print(f"autograd {auto_21} · your numerical gradient {numeric_21} · "
          f"largest gap {np.max(np.abs(auto_21 - numeric_21)):.1e}")
wb.check("5.21b", corner_21)

**Le prix d'un gradient** : la même comparaison sur la fonction de Rosenbrock à 2 000 variables (`valley_21`), dont le code marche aussi bien sur un tableau NumPy que sur un tenseur. Exécute la cellule et compare les deux durées.

In [ ]:
weights_21 = np.linspace(-1.0, 1.0, 2000)          # 2,000 variables


def valley_21(w):
    """Rosenbrock in 2,000 dimensions; the same code works on NumPy arrays and on PyTorch tensors."""
    return (100 * (w[1:] - w[:-1] ** 2) ** 2 + (1 - w[:-1]) ** 2).sum()


with wb.attempt("5.21"):
    mylearn.calculus.numerical_gradient(valley_21, weights_21[:3])      # first, make sure your function is written
    with wb.timer("numerical gradient, 2,000 variables"):
        numeric_big_21 = mylearn.calculus.numerical_gradient(valley_21, weights_21)
    with wb.timer("torch.autograd, 2,000 variables"):
        auto_big_21 = torch_gradient(valley_21, weights_21)
    print(f"largest gap between the two gradients: {np.max(np.abs(numeric_big_21 - auto_big_21)):.1e}")

## Partie D · Dessiner, tester, classer, et un défi

Fiche §5.4 et les pièges classiques. Tu dessines la goutte d'eau du livre sur la vallée de Rosenbrock, tu protèges un gradient numérique par des tests de propriétés, tu classes des points critiques, puis tu règles une descente pour atteindre le fond de la vallée.

### Ex 5.22 — L'eau qui descend la surface (figure 5.18) 🎨 ★★ ⏱️ 30 min
**Objectif :** dessiner une descente de gradient sur une surface en trois dimensions et sur sa carte, avec les directions de plus grande descente.  
**Prérequis :** Ex 5.18 · fiche §5.4 (descendre une surface pas à pas, 🧮 lignes de niveau et flèches) · livre figure 5.18 · **Fil rouge :** Rosenbrock · **Parcours :** C

Le livre montre la descente comme une goutte d'eau qui suit, pas après pas, la direction de plus grande descente (figures 5.16 et 5.18). Refais cette image sur la vallée de Rosenbrock : 3 000 pas de descente depuis $(-1{,}2 ;\ 1)$, le départ classique, avec `lr=0.001` et le gradient exact `rosenbrock_gradient`.

1. Écris `draw_surface_22(ax, path)` : sur l'axe 3D `ax`, la surface sur $[-2, 2] \times [-1, 3]$ (`np.meshgrid`, puis `ax.plot_surface(X, Y, Z, cmap=..., alpha=0.6)`), et par-dessus le chemin, à la hauteur de la surface (`ax.plot(xs, ys, zs)`), départ et arrivée marqués. En hauteur, prends $\log_{10}(1 + f)$ : les parois montent jusqu'à 2 500 et écraseraient la vallée (le logarithme garde les mêmes directions de descente).
2. Écris `draw_map_22(ax, path)` : sur un axe ordinaire, la carte de la même surface (`wb.plot.plot_contour(wb.synth.rosenbrock, ..., ax=ax)` trace les lignes de niveau et le chemin), et, tous les 300 pas, une flèche dans la direction de plus grande descente $-\nabla f$ (`ax.quiver` ; divise chaque flèche par sa norme pour qu'elles aient toutes la même longueur).

La vérification calcule le chemin avec ta fonction `gradient_descent`, contrôle sa forme, puis trace les deux panneaux avec tes fonctions. Compare avec la figure du corrigé, puis réponds dans tes notes : où en est la goutte après 3 000 pas ? Pourquoi descend-elle si vite au début, et si lentement ensuite ?

In [ ]:
START_22 = [-1.2, 1.0]
LR_22, STEPS_22 = 0.001, 3000

In [ ]:
def draw_surface_22(ax, path):
    """The surface log10(1 + f) on [-2, 2] x [-1, 3] (3-D axis ax), with the path drawn on it."""
    raise NotImplementedError("draw_surface_22() is not written yet")


def draw_map_22(ax, path):
    """The map of the same surface, the path, and a unit arrow of -gradient every 300 steps."""
    raise NotImplementedError("draw_map_22() is not written yet")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.22"):
    end_22, path_22 = mylearn.calculus.gradient_descent(rosenbrock_gradient, START_22, lr=LR_22, n_steps=STEPS_22)
    verdict("5.22", np.shape(path_22) == (STEPS_22 + 1, 2),
            f"le chemin a {STEPS_22 + 1} points ; la goutte s'arrête en ({end_22[0]:.3f} ; {end_22[1]:.3f}).",
            "gradient_descent doit renvoyer ici un chemin de forme (3001, 2) : reprends 5.18.")
    fig = plt.figure(figsize=(12, 4.8))
    try:
        draw_surface_22(fig.add_subplot(1, 2, 1, projection="3d"), path_22)
        draw_map_22(fig.add_subplot(1, 2, 2), path_22)
    except BaseException:
        plt.close(fig)                    # no empty figure while the drawing functions are not written
        raise
    plt.tight_layout()
    plt.show()

### Ex 5.23 — Tests de propriétés paramétrés avec pytest 🛠️ ★★ ⏱️ 20 min
**Objectif :** tester une fonction numérique par des propriétés vraies sur beaucoup d'entrées, avec `@pytest.mark.parametrize`.  
**Prérequis :** Ex 5.15 · 0A.62 (pytest, parametrize) · Ex 4.23 · **Parcours :** C

Pour une fonction numérique, on ne connaît pas toujours la valeur exacte attendue ; on connaît en revanche des **propriétés** qu'elle doit respecter sur beaucoup d'entrées. C'est l'idée des **tests de propriétés** : un même test, répété sur plusieurs entrées avec `@pytest.mark.parametrize` (0A.62). Pour un gradient numérique, par exemple :
- sur une forme quadratique $f(\mathbf{x}) = \mathbf{x}^\top A\,\mathbf{x} + \mathbf{b}^\top\mathbf{x}$, le gradient exact est $(A + A^\top)\,\mathbf{x} + \mathbf{b}$ (0B), et la différence centrée le retrouve aux arrondis près ; on peut le vérifier en dimension 1, 2, 5…, avec des $A$, $\mathbf{b}$ et $\mathbf{x}$ tirés au hasard avec une graine ;
- un point écrit en entiers donne le même gradient que le même point écrit en flottants ;
- le gradient a la forme du point, même quand le point est une matrice.

Écris au moins **trois** tests pytest qui appellent `numerical_gradient(f, x)`, dont **au moins un** paramétré par `@pytest.mark.parametrize`, et range-les dans la liste `my_tests_23`. Comme en 4.23, la vérification lance vraiment pytest : tes tests doivent **passer** sur une version juste (`gradient_ok_23`) et **échouer** sur chacune des trois versions buggées de la cellule ci-dessous (lis-les). Pour comparer des flottants, prends une tolérance serrée, `np.allclose(..., atol=1e-6)` par exemple : une tolérance trop large laisserait passer un bug. Tes tests peuvent utiliser `numerical_gradient`, `np`, `math` et `pytest` ; seules tes fonctions de test sont copiées dans le fichier que lance pytest : définis tout ce dont elles ont besoin (une fonction $f$, une matrice) **à l'intérieur** de chaque test.

In [ ]:
def gradient_ok_23(f, x, h=1e-5):
    """A correct numerical gradient, to test your tests."""
    point = np.array(x, dtype=float)
    flat, grad = point.reshape(-1), np.zeros_like(point)
    flat_grad = grad.reshape(-1)
    for i in range(flat.size):
        old = flat[i]
        flat[i] = old + h
        f_plus = f(point)
        flat[i] = old - h
        f_minus = f(point)
        flat[i] = old
        flat_grad[i] = (f_plus - f_minus) / (2 * h)
    return grad


def gradient_bug_1_23(f, x, h=1e-5):
    """Bug 1: the forward difference instead of the central one."""
    point = np.array(x, dtype=float)
    flat, grad = point.reshape(-1), np.zeros_like(point)
    flat_grad = grad.reshape(-1)
    f_here = f(point)
    for i in range(flat.size):
        old = flat[i]
        flat[i] = old + h
        flat_grad[i] = (f(point) - f_here) / h
        flat[i] = old
    return grad


def gradient_bug_2_23(f, x, h=1e-5):
    """Bug 2: copies x but keeps its type, so an integer point stays an integer array."""
    point = np.array(x)
    flat, grad = point.reshape(-1), np.zeros(point.shape)
    flat_grad = grad.reshape(-1)
    for i in range(flat.size):
        old = flat[i]
        flat[i] = old + h
        f_plus = f(point)
        flat[i] = old - h
        f_minus = f(point)
        flat[i] = old
        flat_grad[i] = (f_plus - f_minus) / (2 * h)
    return grad


def gradient_bug_3_23(f, x, h=1e-5):
    """Bug 3: only right for a vector; for a matrix, point[i] is a whole row."""
    point = np.array(x, dtype=float)
    grad = np.zeros(len(point))
    for i in range(len(point)):
        old = point[i].copy()
        point[i] = old + h
        f_plus = f(point)
        point[i] = old - h
        f_minus = f(point)
        point[i] = old
        grad[i] = (f_plus - f_minus) / (2 * h)
    return grad

In [ ]:
# Write your test functions here; each one calls numerical_gradient(f, x).
# @pytest.mark.parametrize("n", [1, 2, 5])
# def test_something(n):
#     ...
#     assert ...

my_tests_23 = ...   # the list of your test functions

In [ ]:
if not filled(my_tests_23):
    print("⏳ Ex 5.23 : tests pas encore écrits.")
else:
    marked_23 = [test for test in my_tests_23
                 if any(mark.name == "parametrize" for mark in getattr(test, "pytestmark", []))]
    verdict("5.23", len(my_tests_23) >= 3 and len(marked_23) >= 1,
            f"{len(my_tests_23)} tests, dont {len(marked_23)} paramétré(s) par @pytest.mark.parametrize.",
            "il faut au moins trois tests, dont au moins un décoré par @pytest.mark.parametrize.")
    good_23 = wb.run_pytest(my_tests_23, subject=gradient_ok_23, name="numerical_gradient")
    verdict("5.23", good_23.ok, f"tes tests passent sur une version juste ({good_23.passed} cas).",
            "tes tests doivent tous passer sur une version juste (voir le détail au-dessus).")
    if good_23.ok:
        for bug in [gradient_bug_1_23, gradient_bug_2_23, gradient_bug_3_23]:
            result = wb.run_pytest(my_tests_23, subject=bug, name="numerical_gradient", quiet=True)
            verdict("5.23", result.failed + result.errors > 0, f"{bug.__name__} est attrapé ({result.failed} cas en échec).",
                    f"tes tests passent sur {bug.__name__}, qui est faux : ajoute un test qui le fait échouer.")

### Ex 5.24 — Minimum, maximum, selle ou plat : classify_critical_point 🔨 ★★★ ⏱️ 35 min
**Objectif :** classer un point critique par les signes des dérivées secondes dans plusieurs directions, sans matrice hessienne.  
**Prérequis :** Ex 5.11, Ex 5.15 · ✏️ 5.5 · fiche §5.4 (les points où le gradient s'annule) · **Fil rouge :** synthétique · **mylearn :** `calculus.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/calculus.py`, mêmes règles qu'en 5.11 (NumPy permis, SciPy et PyTorch non). Enregistre, puis relance la cellule de vérification.

Écris `classify_critical_point(f, x, h=1e-3, tol=1e-6, grad_tol=1e-4)` (lis sa docstring) :
- `x` doit être à une dimension, sinon `ValueError` ; si la norme de `numerical_gradient(f, x)` (ta fonction de 5.15, avec son pas par défaut) dépasse `grad_tol`, `x` n'est pas un point critique : `ValueError` ;
- les directions $\mathbf{d}$ : les axes $\mathbf{e}_i$, puis les diagonales $\mathbf{e}_i + \mathbf{e}_j$ et $\mathbf{e}_i - \mathbf{e}_j$ pour $i < j$ (`np.eye(n)` donne les $\mathbf{e}_i$) ;
- dans chaque direction, la différence seconde $D_\mathbf{d} = \frac{f(\mathbf{x} + h\mathbf{d}) - 2f(\mathbf{x}) + f(\mathbf{x} - h\mathbf{d})}{h^2}$ ; une valeur telle que $|D_\mathbf{d}| \le$ `tol` compte comme nulle ;
- renvoie `"minimum"` si toutes les $D_\mathbf{d}$ sont $>$ `tol`, `"maximum"` si toutes sont $<$ `-tol`, `"saddle"` s'il y en a des deux signes, et `"flat"` sinon (on ne peut pas conclure).

Vérifications sur $h(\mathbf{v}) = v_0^4 + v_1^4 - 4\,v_0 v_1$ (`h_24`, dont les points critiques sont $(0, 0)$, $(1, 1)$ et $(-1, -1)$) et sur $k(\mathbf{v}) = v_0^3 + v_1^2$ (`k_24`). La cellule de vérification appelle ta fonction :
a) en $(0, 0)$, pour $h$ ;
b) en $(1, 1)$, pour $h$ ;
c) en $(0, 0)$, pour $k$ ;
d) le nom de l'erreur levée en $(1, 0)$, pour $h$ ;
puis les tests de `classify_critical_point`.

Dans tes notes : en $(0, 0)$, $h$ ne varie le long des axes qu'en $v^4$ : quelles directions décident du verdict ? Le résultat de c) dit-il que $(0, 0)$ est un minimum de $k$ ? Que vaut $k$ en $(-0{,}1 ;\ 0)$ ?

In [ ]:
def h_24(v):
    """v0⁴ + v1⁴ - 4 v0 v1: critical points (0, 0), (1, 1) and (-1, -1)."""
    return v[0] ** 4 + v[1] ** 4 - 4 * v[0] * v[1]


def k_24(v):
    """v0³ + v1²."""
    return v[0] ** 3 + v[1] ** 2

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.24"):
    wb.check("5.24a", mylearn.calculus.classify_critical_point(h_24, [0.0, 0.0]), computed=True)
    wb.check("5.24b", mylearn.calculus.classify_critical_point(h_24, [1.0, 1.0]), computed=True)
    wb.check("5.24c", mylearn.calculus.classify_critical_point(k_24, [0.0, 0.0]), computed=True)
    wb.check("5.24d", error_name(mylearn.calculus.classify_critical_point, h_24, [1.0, 0.0]), computed=True)
    run_calculus_tests("test_classify_critical_point_")

### Ex 5.25 — Atteindre le fond de la vallée de Rosenbrock 🏆 ★★★ ⏱️ 60 min
**Objectif :** régler un learning rate, ou un petit programme de learning rates, pour atteindre et tenir le minimum d'une vallée mal conditionnée.  
**Prérequis :** Ex 5.18, Ex 5.19 · ∂ 5.7 · fiche §5.4 (Rosenbrock) · **Fil rouge :** Rosenbrock · **Parcours :** C

**Défi.** Pars de $(-1{,}5 ;\ 2)$ et descends la vallée de Rosenbrock ($a = 1$, $b = 100$) jusqu'au minimum $(1, 1)$, avec ta fonction `gradient_descent` et le gradient exact `rosenbrock_gradient`. Écris `schedule_25()`, qui renvoie ton **programme** : une liste de 1 à 3 phases `(lr, n_steps)`, jouées l'une après l'autre, chaque phase repartant du point où la précédente s'est arrêtée. Règles :
- chaque phase compte au moins 100 pas, et le total au plus **10 000 pas** ;
- à la fin, le point est à moins de $10^{-3}$ de $(1, 1)$ (distance euclidienne) ;
- et il y **reste** : 10 000 pas de plus avec le learning rate de la dernière phase ne doivent jamais l'en éloigner de $10^{-3}$ ou plus, et doivent finir à moins de $10^{-4}$ de $(1, 1)$.

Commence par mesurer, avec un seul learning rate : combien de pas faut-il avec 0,001 ? avec 0,0015 ? avec 0,002 ? Pistes : la leçon du bol de 5.19 (le plus grand learning rate possible dépend de la courbure la plus forte) s'applique au fond de la vallée, où la courbure la plus forte vaut environ 1 000 (∂ 5.7) ; et la règle « il y reste » n'est pas là pour rien.

In [ ]:
START_25 = (-1.5, 2.0)                   # a tuple: nothing can change it
TARGET_25 = np.array([1.0, 1.0])


def grade_25(phases, start=(-1.5, 2.0), target=(1.0, 1.0)):
    """Play the phases with YOUR gradient_descent from `start`: (total steps, final distance to `target`, then, during
    10,000 more steps with the last learning rate, the worst distance and the distance at the end)."""
    point, target = np.array(start, dtype=float), np.array(target, dtype=float)
    for lr, n_steps in phases:
        point, _ = mylearn.calculus.gradient_descent(rosenbrock_gradient, np.array(point, dtype=float), lr=lr,
                                                    n_steps=int(n_steps))
    final = float(np.linalg.norm(np.asarray(point, dtype=float) - target))
    _, stay = mylearn.calculus.gradient_descent(rosenbrock_gradient, np.array(point, dtype=float), lr=phases[-1][0],
                                                n_steps=10_000)
    distances = np.linalg.norm(np.asarray(stay, dtype=float) - target, axis=1)
    return sum(int(n_steps) for _, n_steps in phases), final, float(distances.max()), float(distances[-1])

In [ ]:
def schedule_25():
    """Your programme: a list of 1 to 3 phases (lr, n_steps), played one after the other from START_25."""
    raise NotImplementedError("schedule_25() is not written yet")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="5")   # reload your saved file
with wb.attempt("5.25"):
    phases_25 = [tuple(phase) for phase in schedule_25()]
    if not (1 <= len(phases_25) <= 3 and all(len(phase) == 2 and np.ndim(phase[0]) == 0 and np.ndim(phase[1]) == 0
                                                and phase[0] > 0 and float(phase[1]).is_integer() and phase[1] >= 100
                                                for phase in phases_25)):
        verdict("5.25", False, "", "schedule_25() doit renvoyer une liste de 1 à 3 phases (lr, n_steps) : lr un nombre "
                "> 0, n_steps un entier au moins égal à 100.")
    else:
        with np.errstate(over="ignore", invalid="ignore"):
            total_25, final_25, worst_25, end_25 = grade_25(phases_25)
        print(f"{len(phases_25)} phase(s), {total_25} steps: final distance {final_25:.2e}; during the next 10,000 "
              f"steps: worst distance {worst_25:.2e}, distance at the end {end_25:.2e}")
        verdict("5.25", total_25 <= 10_000 and final_25 < 1e-3 and worst_25 < 1e-3 and end_25 < 1e-4,
                f"défi réussi : (1, 1) atteint à {final_25:.1e} près en {total_25} pas, et tenu.",
                "il faut au plus 10 000 pas et une distance finale inférieure à 1e-3 ; puis, pendant les 10 000 pas "
                "suivants, ne jamais s'éloigner à 1e-3 ou plus, et finir à moins de 1e-4.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu estimer une dérivée par différences finies, et expliquer comment choisir le pas $h$ ?
2. Sais-tu calculer un gradient, le lire sur une carte de lignes de niveau, et classer un point critique ?
3. Sais-tu écrire une descente de gradient, et reconnaître un learning rate trop petit ou trop grand ?

**Pour aller plus loin** : les références de la fiche (la vidéo de 3Blue1Brown sur la descente de gradient, l'article de *Distill* sur les vallées mal proportionnées, l'introduction à `torch.autograd`). La suite : le ch. 6 (information et entropie), puis le ch. 18, où `numerical_gradient` vérifiera tes rétropropagations, et le ch. 19, où la vallée de Rosenbrock et `gradient_descent` serviront à comparer les optimiseurs.